# Classical vs. Quantum-Hybrid Ulcer Classifier — full pipeline (GPU)

Runs the group-safe split fix, all three models (fine-tuned classical, frozen+matched-head classical, quantum-hybrid), corrected statistics, and the image+location multimodal fusion experiment, on a free Colab GPU instead of a CPU laptop that sleeps mid-run.

**Before running:** Runtime -> Change runtime type -> T4 GPU. Then Runtime -> Run all.

At the end this prints the final accuracy/statistics summary and downloads a `results_bundle.zip` (all of `results/`, `figures/`, `data/splits/`, `data/metadata/`) to your machine — copy its contents into your local `research/` folder to keep everything in one place.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/shubhisingh1510/AI.git repo
%cd repo
!git log --oneline -3

In [ ]:
# Colab already ships a CUDA-enabled torch that satisfies requirements.txt's torch>=2.1 /
# torchvision>=0.16 -- only install the packages Colab doesn't already have, so we don't
# risk pip silently swapping in a CPU-only torch build.
!pip install -q pennylane pennylane-lightning grad-cam opencv-python-headless

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU not enabled -- set Runtime > Change runtime type > T4 GPU, then Runtime > Restart and run all.'

## 1. Fetch the data and build the group-safe split

In [ ]:
!python scripts/fetch_azh_dataset.py --config configs/config.yaml
!python scripts/fetch_medetec_supplement.py --config configs/config.yaml

In [ ]:
# Original stratified split (kept for reference/backward compat with old results).
!python src/data_prep.py --config configs/config.yaml

In [ ]:
# The corrected, group-safe split: fixes the cross-class filename-collision bug and
# keeps near-duplicate images out of both sides of train/test. This is the split used
# by every cell below (via --splits-metadata run_metadata_groupsafe.json).
!python src/dedupe_and_group_split.py --config configs/config.yaml

In [ ]:
!python src/dataset_audit.py --config configs/config.yaml

## 2. Classical ResNet-50 baseline (group-safe split, regularized)

GPU -- this is the step that was taking hours on CPU. Includes head dropout, label smoothing, and gradual per-block unfreezing with discriminative learning rates (configs/config.yaml), targeting the train/val overfitting gap seen in the original run.

In [ ]:
!python src/classical_baseline.py --config configs/config.yaml \
    --splits-metadata run_metadata_groupsafe.json --output-suffix _groupsafe

## 3. Matched classical control

Same frozen backbone + PCA-6 input as the quantum model, but a classical head with a parameter count matched to the quantum circuit -- isolates "frozen features generalize better" from "the quantum circuit specifically helps."

In [ ]:
!python src/classical_frozen_head.py --config configs/config.yaml --head-type mlp \
    --splits-metadata run_metadata_groupsafe.json --output-suffix _groupsafe

## 4. Quantum-hybrid classifier

This part is CPU-bound regardless of GPU (PennyLane's `lightning.qubit` state-vector simulator runs on CPU), but Colab's CPU won't fall asleep mid-run the way a laptop does.

In [ ]:
!python src/quantum_hybrid.py --config configs/config.yaml \
    --splits-metadata run_metadata_groupsafe.json --output-suffix _groupsafe

## 5. Compare all three models + corrected statistics

Reports the naive paired t-test AND the Nadeau-Bengio corrected resampled t-test side by side (the corrected one is the more defensible claim -- see evaluate_compare.py's docstring) plus McNemar's, for every pair of models.

In [ ]:
!python src/evaluate_compare.py --config configs/config.yaml \
    --input-suffix _groupsafe --models classical,frozen_head,quantum

## 6. Image + wound-location multimodal fusion

Scoped to the 730 original AZH images that have real wound-location labels (the 161-image Medetec supplement has none) -- a separate experiment from the main numbers above, not a replacement for them.

In [ ]:
!python src/multimodal_fusion.py --config configs/config.yaml \
    --backbone-suffix _groupsafe --output-suffix _groupsafe

## 7. (Optional, slow) Full ablation sweep with the data re-uploading axis

Sweeps qubit count x circuit depth x {single-injection, data re-uploading} x {pca, domain encoding} -- this is the most expensive cell in the notebook (many small training runs). Set `RUN_ABLATION = True` below to include it; skip it for a faster pass and come back to it once the numbers above look right.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    !python src/ablation.py --config configs/config.yaml

## 8. Print the final numbers

In [ ]:
import json

def show(path):
    try:
        with open(path) as f:
            d = json.load(f)
    except FileNotFoundError:
        print(f'--- {path} --- (not found, skipping)')
        return
    print(f'--- {path} ---')
    print(json.dumps(d, indent=2)[:3000])
    print()

show('results/classical_metrics_groupsafe.json')
show('results/classical_frozen_head_metrics_groupsafe.json')
show('results/quantum_metrics_groupsafe.json')
show('results/statistical_tests_groupsafe.json')
show('results/multimodal_fusion_metrics.json')

import pandas as pd
try:
    print('--- results/comparison_table_groupsafe.csv ---')
    print(pd.read_csv('results/comparison_table_groupsafe.csv').to_string(index=False))
except FileNotFoundError:
    pass

## 9. Bundle everything and download it to your machine

Copy the unzipped contents into your local `research/` folder (matching subfolder structure) so committed history, figures, and results all line up. The report (`paper/report.html`) is a separate follow-up step once you've reviewed these numbers -- it needs updating for the new three-model comparison and isn't regenerated here.

In [ ]:
import os, zipfile
with zipfile.ZipFile('/content/results_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['results', 'figures', 'data/splits', 'data/metadata']:
        for root, _, files in os.walk(folder):
            for fn in files:
                fp = os.path.join(root, fn)
                zf.write(fp, fp)

from google.colab import files
files.download('/content/results_bundle.zip')